(C) Copyright 2025 - ECMWF and individual contributors.

This software is licensed under the terms of the Apache Licence Version 2.0
which can be obtained at http://www.apache.org/licenses/LICENSE-2.0.
In applying this licence, ECMWF does not waive the privileges and immunities
granted to it by virtue of its status as an intergovernmental organisation nor
does it submit to any jurisdiction.

# POF Forecast

Importing all the necessary python libraries

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import cftime
from xarray.coding.times import CFDatetimeCoder
import os
import cdsapi
import zipfile
import datetime
from ecmwf.opendata import Client
import requests
import tarfile
from contextlib import ExitStack
from pof_utils import (
    PARAM_IDS,
    field_at,
    first_variable,
    forecast_daily_means,
    grib_filter,
    open_grib,
    read_grib,
    regrid_with_cdo,
    retrieve_ecbox_file,
    time_dim,
    to_training_grid,
    write_grib_dataset,
)


Configuration Options

Here we define the paths for files and the dates we will use for our forecast.

We also load the model we created in the Trainer notebook


In [ ]:
# Folder structure
base_path = Path("./data/")
os.makedirs("./outputs/", exist_ok=True)

# Dates of Forecast
# Today's 00z run is not complete on the open data server until later in the day, so use yesterday's
run_date = datetime.date.today() - datetime.timedelta(days=1)
forecast_start_date = run_date
run_time = 0             # 00z run
forecast_days = 5        # number of daily (00 UTC) forecast steps to retrieve
year = run_date.year
years=[year]
months = [run_date.month]
month=run_date.month

cds_months=[f'{m:02d}' for m in months]

# Where we will store our predictions
output_file = f"./outputs/POF_prediction_{year}_{months[0]:02d}.grib2"

# Load trained model
model = joblib.load("./data/POF_model.joblib")

ECMWF Open Data Retrieval

This code block retrieves the IFS deterministic forecast (`oper`) from ECMWF Open Data at 0.1° resolution (`0p1`), which is the same resolution as ERA5-Land used for training. No API key is needed.

We retrieve 2m temperature, 2m dewpoint, 10m U and V wind components and total precipitation every 3 hours, so that we can calculate the same daily means the model was trained on. Relative humidity and wind speed are not available in Open Data, so we calculate them for each step as we did in the Data Generator. Total precipitation is accumulated from the start of the forecast, so we de-accumulate it over each calendar day (00 to 24 UTC) and divide by 24 to give a mean hourly amount, exactly as we do for ERA5-Land in the Data Generator.

Open Data runs longitude from -180 to 180 while the training data runs from 0 to 360, so the daily means are moved onto the training grid. They are written to a single GRIB2 file used by the prediction step.

Open Data is licensed under CC BY 4.0; please attribute ECMWF.

In [ ]:
client = Client(source="ecmwf", model="ifs", resol="0p1", preserve_request_order=False,
    infer_stream_keyword=True)
steps = [24 * d for d in range(forecast_days + 1)]  # daily 00 UTC steps for the fuel moisture models
three_hourly_steps = list(range(0, 24 * (forecast_days + 1) + 1, 3))  # every 3 hours for the daily means

def to_era5_land_grid(da):
    # Open Data uses longitude -180..180, ERA5-Land and the climate files use 0..360
    da = da.assign_coords(longitude=da.longitude.round(4) % 360).sortby("longitude")
    da = da.swap_dims(step="valid_time")
    return da.drop_vars([c for c in da.coords if c not in ("valid_time", "latitude", "longitude")])

for shortname, param, var in [["T2M", "2t", "t2m"],
                                ["D2M", "2d", "d2m"],
                                ["SKT", "skt", "skt"],
                                ["SP", "sp", "sp"],
                                ["10U", "10u", "u10"],
                                ["10V", "10v", "v10"],
                                ["P", "tp", "tp"]]:

    grib_file = f"data/{shortname}_{run_date.strftime('%Y%m%d')}.grib2"
    if not os.path.exists(grib_file):
        req = {
            "date": run_date,
            "time": run_time,
            "type": "fc",
            "levtype": "sfc",
            "step": three_hourly_steps,
            "param": param,
        }
        print(req)
        client.retrieve(req, grib_file)

for shortname, param, level in [["VSW1", "39",1],
                                ["VSW2", "40", 2],
                                ["VSW3", "41", 3],
                                ["VSW4", "42", 4]]:

    grib_file = f"data/{shortname}_{run_date.strftime('%Y%m%d')}.grib2"
    if not os.path.exists(grib_file):
        req = {
            "date": run_date,
            "time": run_time,
            "type": "fc",
            "levtype": "sol",
            "levelist": level,
            "step": steps,
            "param": "vsw",
        }
        client.retrieve(req, grib_file)

daily_file = f"data/FORCING_DAILY_{run_date.strftime('%Y%m%d')}.grib2"
if not os.path.exists(daily_file):
    with ExitStack() as stack:
        three_hourly = {
            name: first_variable(stack.enter_context(open_grib(f"data/{shortname}_{run_date.strftime('%Y%m%d')}.grib2")))
            for name, shortname in [["t2m", "T2M"], ["d2m", "D2M"], ["u10", "10U"], ["v10", "10V"], ["tp", "P"]]
        }
        daily = forecast_daily_means(**three_hourly, run_date=run_date, forecast_days=forecast_days)
    write_grib_dataset(
        daily,
        daily_file,
        PARAM_IDS,
        (run_date + datetime.timedelta(days=1)).strftime("%Y%m%d"),
    )


In [ ]:
# Retrieve Climate from OpenIFS
# https://sites.ecmwf.int/openifs/openifs-download/ifsdata/

# https://sites.ecmwf.int/openifs/openifs-download/ifsdata/48r1/climate.v020/48r1_climate.v020_1279.tar.gz

filename="48r1_climate.v020_1279.tar.gz"
    
output_path=f"data/CLIMATE/1279_4"
if not os.path.exists(output_path):
    base_url = "https://sites.ecmwf.int/openifs/openifs-download/ifsdata/48r1/climate.v020/"

    response = requests.get(f'{base_url}/{filename}',)

    if response.status_code == 200:
        with open(f"data/{filename}", "wb") as f:
            f.write(response.content)
        print("Downloaded!")
    else:
        print(response.status_code, response.text)


    archive_path = Path(f"data/{filename}")
    climate_dir = Path("data/CLIMATE")
    climate_dir.mkdir(parents=True, exist_ok=True)

    with tarfile.open(archive_path, "r:gz") as archive:
        archive.extractall(climate_dir)


In [ ]:
os.makedirs("data", exist_ok=True)
os.makedirs("data/CLIMATE", exist_ok=True)
os.makedirs("data/DFMC_INI", exist_ok=True)
for month in range(1, 13):
    retrieve_ecbox_file(f"DFMC_INI/DFMC1_INI_2018_{month:02d}.grib2")
    retrieve_ecbox_file(f"DFMC_INI/DFMC10_INI_2018_{month:02d}.grib2")
    retrieve_ecbox_file(f"DFMC_INI/DFMC100_INI_2018_{month:02d}.grib2")
    retrieve_ecbox_file(f"DFMC_INI/DFMC1000_INI_2018_{month:02d}.grib2")
retrieve_ecbox_file(f"CLIMATE/DM_FUEL.grib2")
retrieve_ecbox_file(f"CLIMATE/STATIC_FUEL_FRACTIONS.grib2")
retrieve_ecbox_file("CLIMATE/POP_2020.grib")
retrieve_ecbox_file("CLIMATE/road_density_2015_agg_r.grib")

In [ ]:
# Calculate LFMC


date_string = forecast_start_date.strftime('%Y_%m_%d')
forecast_date = forecast_start_date.strftime("%Y%m%d")
t2_grid_file = Path(f"data/T2M_{forecast_date}.grib2")


############################################################
# READ STATIC FRACTION MAP
############################################################
print("Reading Static Fraction Map")

static_fraction_backend = {
    "indexpath": "",
    "filter_by_keys": {"discipline": 2, "parameterCategory": 4, "parameterNumber": 37},
}
with xr.open_dataset(
    "data/CLIMATE/STATIC_FUEL_FRACTIONS.grib2",
    decode_times=False,
    engine="cfgrib",
    backend_kwargs=static_fraction_backend,
) as static_low:
    static_live_low = np.squeeze(
        static_low[next(iter(static_low.data_vars))].load().values
    )

static_fraction_backend["filter_by_keys"]["parameterNumber"] = 38
with xr.open_dataset(
    "data/CLIMATE/STATIC_FUEL_FRACTIONS.grib2",
    decode_times=False,
    engine="cfgrib",
    backend_kwargs=static_fraction_backend,
) as static_high:
    static_live_high = np.squeeze(
        static_high[next(iter(static_high.data_vars))].load().values
    )

lf = static_live_low/(static_live_low+static_live_high)
hf = 1.0-lf

tvls_grib = regrid_with_cdo(
    "data/CLIMATE/1279_4/tvl",
    "data/CLIMATE/tvl_T2_GRID.grib2",
    t2_grid_file,
    "remapnn",
)
with xr.open_dataset(tvls_grib, engine="cfgrib", decode_times=False, backend_kwargs={"indexpath": ""}) as tvls_ds:
    tvls = np.squeeze(tvls_ds.tvl.load().values)

tvhs_grib = regrid_with_cdo(
    "data/CLIMATE/1279_4/tvh",
    "data/CLIMATE/tvh_T2_GRID.grib2",
    t2_grid_file,
    "remapnn",
)
with xr.open_dataset(tvhs_grib, engine="cfgrib", decode_times=False, backend_kwargs={"indexpath": ""}) as tvhs_ds:
    lat = tvhs_ds.latitude.values
    lon = tvhs_ds.longitude.values
    tvhs = np.squeeze(tvhs_ds.tvh.load().values)

lail_grib = regrid_with_cdo(
    "data/CLIMATE/1279_4/month_lail",
    "data/CLIMATE/month_lail_T2_GRID.grib2",
    t2_grid_file,
    "remapnn",
)
with xr.open_dataset(
    lail_grib,
    decode_times=False,
    engine="cfgrib",
    backend_kwargs={"indexpath": "", "filter_by_keys": {"shortName": "lai_lv"}},
) as lail_ds:
    lails = lail_ds.lai_lv.load().values

laih_grib = regrid_with_cdo(
    "data/CLIMATE/1279_4/month_laih",
    "data/CLIMATE/month_laih_T2_GRID.grib2",
    t2_grid_file,
    "remapnn",
)
with xr.open_dataset(
    laih_grib,
    decode_times=False,
    engine="cfgrib",
    backend_kwargs={"indexpath": "", "filter_by_keys": {"shortName": "lai_hv"}},
) as laih_ds:
    laihs = laih_ds.lai_hv.load().values

############################################################
# DECLARE COEFFICIENTS
############################################################
print("Declaring Coeffients")
A=np.array([110.75996087,197.41836383,132.58990975,132.58990975,131.54113182,131.54113182,165.76200312,242.31794467,242.31794467,110.75996087,242.31794467,242.31794467,242.31794467,242.31794467,242.31794467,158.77301792,158.77301792,242.31794467,242.31794467,242.31794467])
B=np.array([1500.93002374,119.74072395,65.24821118,65.24821118,78.71811202,78.71811202,148.21931888,237.92003039,237.92003039,1500.93002374,237.92003039,237.92003039,237.92003039,237.92003039,237.92003039,131.02067658,131.02067658,237.92003039,237.92003039,237.92003039])
C=np.array([2.71542147e+01,2.34354649e-01,1.90227693e+00,1.90227693e+00,1.45974849e-14,1.45974849e-14,4.30551321e+00,1.77424187e+00,1.77424187e+00,2.71542147e+01,1.77424187e+00,1.77424187e+00,1.77424187e+00,1.77424187e+00,1.77424187e+00,3.50411009e+00,3.50411009e+00,1.77424187e+00,1.77424187e+00,1.77424187e+00])
D=np.array([4.47694802e-24,2.26407223e-01,3.36086293e-11,3.36086293e-11,1.21311078e-20,1.21311078e-20,3.22205405e-12,3.26481962e-02,3.26481962e-02,4.47694802e-24,3.26481962e-02,3.26481962e-02,3.26481962e-02,3.26481962e-02,3.26481962e-02,1.23468166e-10,1.23468166e-10,3.26481962e-02,3.26481962e-02])
E=np.array([2.20718082e-15,5.28906485e-01,1.23700171e+00,1.23700171e+00,3.82763156e+00,3.82763156e+00,5.70913798e-01,6.27494910e-12,6.27494910e-12,2.20718082e-15,6.27494910e-12,6.27494910e-12,6.27494910e-12,6.27494910e-12,6.27494910e-12,1.72287521e+00,1.72287521e+00,6.27494910e-12,6.27494910e-12])

lows = np.array([1,2,7,8,9,10,11,13,16,17])
highs = np.array([3,4,5,6,18,19])

FRACTION_1 = np.array([24, 35, 26, 26, 24, 25, 27, 0, 47, 24, 17, 0, 25, 0, 0, 23, 23, 19, 19])
FRACTION_2 = np.array([41, 38, 39, 38, 38, 34, 27, 0, 45, 41, 31, 0, 34, 0, 0, 36, 36, 35, 35])
FRACTION_3 = np.array([31, 23, 29, 29, 31, 27, 27, 0, 8, 31, 33, 0, 27, 0, 0, 30, 30, 36, 36])
FRACTION_4 = np.array([4, 4, 6, 7, 7, 14, 19, 0, 0, 4, 19, 0, 14, 0, 0, 11, 11, 10, 10])

############################################################
# READ FORCING
############################################################
print("Reading Forcings")
soil_moisture = []
for layer, shortname in enumerate(("VSW1", "VSW2", "VSW3", "VSW4"), start=1):
    grib_file = f"data/{shortname}_{forecast_date}.grib2"
    with xr.open_dataset(
        grib_file,
        engine="cfgrib",
        backend_kwargs={"indexpath": ""},
    ) as ds_vsw:
        soil_moisture.append(
            np.squeeze(ds_vsw[next(iter(ds_vsw.data_vars))].load().values)
        )
        if layer == 1:
            sm_grib_file = grib_file
            lat = ds_vsw.latitude.values
            lon = ds_vsw.longitude.values
            time = np.atleast_1d(ds_vsw.valid_time.values)

sm1, sm2, sm3, sm4 = soil_moisture

sm1[sm1==0]=1e-7
sm2[sm2==0]=1e-7
sm3[sm3==0]=1e-7
sm4[sm4==0]=1e-7

fm = np.zeros_like(sm1)
fm = np.squeeze(fm)
fml = np.zeros_like(sm1)
fml = np.squeeze(fml)
fmh = np.zeros_like(sm1)
fmh = np.squeeze(fmh)

ind = forecast_start_date.month-1

for j in np.arange(len(time)):
    print(f"Processing day {j}")
    x = np.zeros_like(fm[0,:,:])
    xl = np.zeros_like(fm[0,:,:])
    xh = np.zeros_like(fm[0,:,:])
    s1 = sm1[j,:,:]
    s2 = sm2[j,:,:]
    s3 = sm3[j,:,:]
    s4 = sm4[j,:,:]
    lh = laihs[ind,:,:]
    ll = lails[ind,:,:]

    l_w = np.zeros_like(s1)
    for i in np.arange(len(lows)):
        ind = tvls==lows[i]
        z = lows[i]-1
        l_w[ind] = ( FRACTION_1[z] * s1[ind] + FRACTION_2[z] * s2[ind] + FRACTION_3[z] * s3[ind] + FRACTION_4[z] * s4[ind] )/100

    h_w = np.zeros_like(s1)
    for i in np.arange(len(highs)):
        ind = tvhs==highs[i]
        z = highs[i]-1
        h_w[ind] = ( FRACTION_1[z] * s1[ind] + FRACTION_2[z] * s2[ind] + FRACTION_3[z] * s3[ind] + FRACTION_4[z] * s4[ind] )/100
    for i in np.arange (len(lows)):
        ind = tvls==lows[i]
        inc = lows[i]-1

        x[ind]=x[ind]+lf[ind]*(A[inc]-B[inc]*np.exp(-(C[inc]*l_w[ind]+D[inc]*ll[ind]+E[inc]*l_w[ind]*ll[ind])))
        xl[ind]=(A[inc]-B[inc]*np.exp(-(C[inc]*l_w[ind]+D[inc]*ll[ind]+E[inc]*l_w[ind]*ll[ind])))

    for i in np.arange (len(highs)):
        ind = tvhs==highs[i]
        inc = highs[i]-1

        x[ind]=x[ind]+hf[ind]*(A[inc]-B[inc]*np.exp(-(C[inc]*h_w[ind]+D[inc]*lh[ind]+E[inc]*h_w[ind]*lh[ind])))
        xh[ind]=(A[inc]-B[inc]*np.exp(-(C[inc]*h_w[ind]+D[inc]*lh[ind]+E[inc]*h_w[ind]*lh[ind])))

    fm[j,:,:] = x
    fml[j,:,:] = xl
    fmh[j,:,:] = xh

    forecast_day = forecast_start_date + datetime.timedelta(days=int(j) + 1)
    smonth = forecast_day.month
    ind = smonth-1

fm[fm<0.0]=0.0
fml[fml<0.0]=0.0
fmh[fmh<0.0]=0.0
ds = xr.Dataset({})
ds = ds.assign_coords(time=time)
ds = ds.assign_coords(lat=lat)
ds = ds.assign_coords(lon=lon)
ds.lon.attrs['units'] = 'degrees_east'
ds.lon.attrs['standard_name'] = 'longitude'
ds.lon.attrs['long_name'] = 'Longitude'

ds.lat.attrs['units'] = 'degrees_north'
ds.lat.attrs['standard_name'] = 'latitude'
ds.lat.attrs['long_name'] = 'Latitude'
print(f'Hours since {forecast_start_date.strftime("%Y-%m-%d")} 00:00:00')
ds.time.attrs['standard_name'] = 'time'
ds.time.attrs['long_name'] = 'Time'
ds.time.attrs['units'] = f'Hours since {forecast_start_date.strftime("%Y-%m-%d")} 00:00:00'
ds.time.attrs['calendar'] = 'standard'

ds = ds.assign(LFMC=(['time','lat','lon'],fm))
ds.LFMC.attrs['units'] = '%'
ds = ds.assign(LFMC_low=(['time','lat','lon'],fml))
ds.LFMC.attrs['units'] = '%'
ds = ds.assign(LFMC_high=(['time','lat','lon'],fmh))
ds.LFMC.attrs['units'] = '%'

dm = ds.mean(dim="time")
print(f"Saving GRIB outputs for {date_string}")
write_grib_dataset(
    ds,
    f"data/LFMC_MAP_{date_string}_FC.grib2",
    {"LFMC": 1, "LFMC_low": 2, "LFMC_high": 3},
    date_string.replace("_", ""),
    template_path=sm_grib_file,
)
write_grib_dataset(
    dm,
    f"data/LFMC_timemean_{date_string}_FC.grib2",
    {"LFMC": 1, "LFMC_low": 2, "LFMC_high": 3},
    date_string.replace("_", ""),
    template_path=sm_grib_file,
)

In [ ]:
# Calculate DFMC

forecast_date = forecast_start_date.strftime("%Y%m%d")
forecast_start = forecast_start_date
date_string = forecast_start.strftime("%Y_%m_%d")
mon = forecast_start.strftime("%m")
surface_template = f"data/SKT_{forecast_date}.grib2"


def read_dfmc_restart(path, parameter_number):
    field = read_grib(
        path,
        {"discipline": 0, "parameterCategory": 191, "parameterNumber": parameter_number},
    )
    return np.squeeze(field.values)

print(forecast_start)

############################################################
# READ INITIAL CONDITIONS
############################################################

ini_paths = {
    "1h": f"data/DFMC1_INI_{date_string}.grib2",
    "10h": f"data/DFMC10_INI_{date_string}.grib2",
    "100h": f"data/DFMC100_INI_{date_string}.grib2",
    "1000h": f"data/DFMC1000_INI_{date_string}.grib2",
}
if all(os.path.exists(path) for path in ini_paths.values()):
    print(f"Found GRIB initial conditions for {date_string}")
    ini_1h = read_dfmc_restart(ini_paths["1h"], 6)
    ini_10h = read_dfmc_restart(ini_paths["10h"], 7)
    ini_100h = read_dfmc_restart(ini_paths["100h"], 8)
    ini_1000h = read_dfmc_restart(ini_paths["1000h"], 9)
else:
    print(f"Using climate GRIB2 initial conditions for 2018-{mon}")
    ini_1h = np.squeeze(read_grib(
        f"data/DFMC_INI/DFMC1_INI_2018_{mon}.grib2", {}, decode_times=False,
    ).values)
    ini_10h = np.squeeze(read_grib(
        f"data/DFMC_INI/DFMC10_INI_2018_{mon}.grib2", {}, decode_times=False,
    ).values)
    ini_100h = np.squeeze(read_grib(
        f"data/DFMC_INI/DFMC100_INI_2018_{mon}.grib2", {}, decode_times=False,
    ).values)
    ini_1000h = np.squeeze(read_grib(
        f"data/DFMC_INI/DFMC1000_INI_2018_{mon}.grib2", {}, decode_times=False,
    ).values)

############################################################
# READ STATIC FRACTION MAP
############################################################

static_fraction_path = "data/CLIMATE/STATIC_FUEL_FRACTIONS.grib2"
static_low = np.squeeze(
    read_grib(
        static_fraction_path,
        {"discipline": 2, "parameterCategory": 4, "parameterNumber": 37},
        decode_times=False,
    ).values
)
static_high_raw = np.squeeze(
    read_grib(
        static_fraction_path,
        {"discipline": 2, "parameterCategory": 4, "parameterNumber": 38},
        decode_times=False,
    ).values
)
static_total = static_low + static_high_raw
static_low = np.divide(static_low, static_total, out=np.zeros_like(static_low), where=static_total != 0)
static_high = 1.0 - static_low

tvls = np.squeeze(
    read_grib("data/CLIMATE/tvl_T2_GRID.grib2", {"shortName": "tvl"}, "tvl", decode_times=False).values
)
tvhs = np.squeeze(
    read_grib("data/CLIMATE/tvh_T2_GRID.grib2", {"shortName": "tvh"}, "tvh", decode_times=False).values
)
fuel_type = np.zeros((4,) + tvhs.shape)

############################################################
# STATIC FRACTION COEFFICIENTS
############################################################
print("Static Fraction Coefficients")

t_1h_f = [1.0, 0.5, 1.0, 0.53, 0.53, 0.53, 0.53, 0.5, 0.65, 0.65, 0.5, 0.65, 1.0, 0.65, 1.0, 1.0, 0.65, 0.65, 0.5, 0.5, 0.5]
t_10h_f = [0.0, 0.5, 0.0, 0.47, 0.47, 0.47, 0.47, 0.5, 0.35, 0.35, 0.5, 0.35, 0.0, 0.35, 0.0, 0.0, 0.35, 0.35, 0.5, 0.5, 0.5]
t_100h_w = [1.0, 1.0, 1.0, 0.3, 0.3, 0.3, 0.3, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.5, 0.5, 1.0]
t_1000h_w = [0.0, 0.0, 0.0, 0.7, 0.7, 0.7, 0.7, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 0.5, 0.0]

lows = [0, 1, 2, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 20]
highs = [3, 4, 5, 6, 18, 19]

for fuel_class in lows:
    mask = tvls == fuel_class
    fuel_type[0, mask] += t_1h_f[fuel_class] * static_low[mask]
    fuel_type[1, mask] += t_10h_f[fuel_class] * static_low[mask]
    fuel_type[2, mask] += t_100h_w[fuel_class] * static_low[mask]
    fuel_type[3, mask] += t_1000h_w[fuel_class] * static_low[mask]

for fuel_class in highs:
    mask = tvhs == fuel_class
    fuel_type[0, mask] += t_1h_f[fuel_class] * static_high[mask]
    fuel_type[1, mask] += t_10h_f[fuel_class] * static_high[mask]
    fuel_type[2, mask] += t_100h_w[fuel_class] * static_high[mask]
    fuel_type[3, mask] += t_1000h_w[fuel_class] * static_high[mask]

############################################################
# READ FORECAST GRIB FORCINGS
############################################################

daily_steps = {"step": pd.to_timedelta(steps, unit="h")}
Ta = read_grib(f"data/T2M_{forecast_date}.grib2", {"shortName": "2t"}, "t2m", select=daily_steps)
Td = read_grib(f"data/D2M_{forecast_date}.grib2", {"shortName": "2d"}, "d2m", select=daily_steps)
Ts = read_grib(f"data/SKT_{forecast_date}.grib2", {"shortName": "skt"}, "skt", select=daily_steps)
Pb = read_grib(f"data/SP_{forecast_date}.grib2", {"shortName": "sp"}, "sp", select=daily_steps) / 100.0
precipitation = read_grib(f"data/P_{forecast_date}.grib2", {"shortName": "tp"}, "tp", select=daily_steps)
P = precipitation.diff("step").clip(min=0) * (1000.0 / np.pi)

time = np.atleast_1d(Ta.valid_time.values)
lat = Ta.latitude.values
lon = Ta.longitude.values
if P.sizes["step"] != len(time) - 1:
    raise ValueError("Daily precipitation steps do not match the temperature forecast steps")

Ta = Ta - 273.15
Td = Td - 273.15
Ts = Ts - 273.15
RH = np.exp(17.625 * Td / (243.04 + Td)) / np.exp(17.625 * Ta / (243.04 + Ta))

############################################################
# SETUP VARIABLES
############################################################

A = 0.000772
hc = 1.59 * 10**4
hc = hc / 3600
pr = 0.7
sc = 0.58
top = 0.662 * hc * (pr / sc) ** 0.667
rho = 400
ca = 1005
a1 = 0.0020
a10 = 0.0064
a100 = 0.02
a1000 = 0.064
bot1 = rho * ca * a1
bot10 = rho * ca * a10
bot100 = rho * ca * a100
bot1000 = rho * ca * a1000

############################################################
# FOLIAGE AND WOOD
############################################################
print("DFMC Foliage and Wood")
DFMC1 = np.zeros(np.shape(Ta))
DFMC1[0, :, :] = ini_1h
DFMC10 = np.zeros(np.shape(Ta))
DFMC10[0, :, :] = ini_10h
DFMC100 = np.zeros(np.shape(Ta))
DFMC100[0, :, :] = ini_100h
DFMC1000 = np.zeros(np.shape(Ta))
DFMC1000[0, :, :] = ini_1000h

for i in np.arange(len(time) - 1) + 1:
    Pa = 6.1094 * np.exp((17.625 * Ta[i - 1, :, :]) / (Ta[i - 1, :, :] + 243.04))
    Pd = 6.1094 * np.exp((17.625 * Td[i - 1, :, :]) / (Td[i - 1, :, :] + 243.04))
    Qv = 56720 - (42.8 * (Ts[i - 1, :, :].values) + 273.15)
    Qw = 4000 * np.exp(-14 * DFMC1[i - 1, :, :])
    Co = Qv / (Qv + Qw)
    ps = (Pa * RH[i - 1, :, :] + Pb[i - 1, :, :] * A * (Ts[i - 1, :, :].values - Ta[i - 1, :, :].values)) * Co
    y = (24 / 24) * (0.32 * (1 - np.exp(-1.885 * P[i - 1, :, :])))
    y = 3.0 * y.values
    y[y > 0.85] = 0.85
    y[y < 0.0] = 0.0
    DFMC1[i, :, :] = DFMC1[i - 1, :, :] + y - 2 * (86400 * (top / (Pb[i - 1, :, :] * bot1)) * (ps - Pd))
    DFMC1[i, :, :][DFMC1[i, :, :] > 0.8] = 0.8
    DFMC1[i, :, :][DFMC1[i, :, :] < 0.0] = 0.0

    Qw = 5000 * np.exp(-14 * DFMC10[i - 1, :, :])
    Co = Qv / (Qv + Qw)
    ps = (Pa * RH[i - 1, :, :] + Pb[i - 1, :, :] * A * (Ts[i - 1, :, :].values - Ta[i - 1, :, :].values)) * Co
    y = (24 / 24) * (0.32 * (1 - np.exp(-1.885 * P[i - 1, :, :])))
    y = y.values
    y[y > 0.6] = 0.6
    y[y < 0.0] = 0.0
    DFMC10[i, :, :] = DFMC10[i - 1, :, :] + y - 2 * (86400 * (top / (Pb[i - 1, :, :] * bot10)) * (ps - Pd))
    DFMC10[i, :, :][DFMC10[i, :, :] > 0.6] = 0.6
    DFMC10[i, :, :][DFMC10[i, :, :] < 0.0] = 0.0

    Qw = 6000 * np.exp(-14 * DFMC100[i - 1, :, :])
    Co = Qv / (Qv + Qw)
    ps = (Pa * RH[i - 1, :, :] + Pb[i - 1, :, :] * A * (Ts[i - 1, :, :].values - Ta[i - 1, :, :].values)) * Co
    y = (24 / 24) * (0.32 * (1 - np.exp(-1.885 * P[i - 1, :, :])))
    y = 0.33 * y.values
    y[y > 0.4] = 0.4
    y[y < 0.0] = 0.0
    DFMC100[i, :, :] = DFMC100[i - 1, :, :] + y - 2 * (86400 * (top / (Pb[i - 1, :, :] * bot100)) * (ps - Pd))
    DFMC100[i, :, :][DFMC100[i, :, :] > 0.5] = 0.5
    DFMC100[i, :, :][DFMC100[i, :, :] < 0.0] = 0.0

    Qw = 7000 * np.exp(-14 * DFMC1000[i - 1, :, :])
    Co = Qv / (Qv + Qw)
    ps = (Pa * RH[i - 1, :, :] + Pb[i - 1, :, :] * A * (Ts[i - 1, :, :].values - Ta[i - 1, :, :].values)) * Co
    y = (24 / 24) * (0.32 * (1 - np.exp(-1.885 * P[i - 1, :, :])))
    y = 0.13 * y.values
    y[y > 0.32] = 0.32
    y[y < 0.0] = 0.0
    DFMC1000[i, :, :] = DFMC1000[i - 1, :, :] + y - 2 * (86400 * (top / (Pb[i - 1, :, :] * bot1000)) * (ps - Pd))
    DFMC1000[i, :, :][DFMC1000[i, :, :] > 0.4] = 0.4
    DFMC1000[i, :, :][DFMC1000[i, :, :] < 0.0] = 0.0

DFMC1 = np.nan_to_num(DFMC1, copy=False, nan=0.0)
DFMC10 = np.nan_to_num(DFMC10, copy=False, nan=0.0)
DFMC100 = np.nan_to_num(DFMC100, copy=False, nan=0.0)
DFMC1000 = np.nan_to_num(DFMC1000, copy=False, nan=0.0)

DFMC_FOLIAGE = DFMC1 * fuel_type[0, :, :] + DFMC10 * fuel_type[1, :, :]
DFMC_FOLIAGE[DFMC_FOLIAGE < 0.0] = 0.001
DFMC_WOOD = DFMC100 * fuel_type[2, :, :] + DFMC1000 * fuel_type[3, :, :]
DFMC_WOOD[DFMC_WOOD < 0.0] = 0.001

print(time)
ds = xr.Dataset(
    {
        "DFMC_Foliage": (("time", "lat", "lon"), DFMC_FOLIAGE),
        "DFMC_Wood": (("time", "lat", "lon"), DFMC_WOOD),
    },
    coords={"time": time, "lat": lat, "lon": lon},
)
dm = ds.mean(dim="time")
print(f"Saving DFMC GRIB2 maps for {date_string}")
write_grib_dataset(
    ds,
    f"data/DFMC_MAP_{date_string}_FC.grib2",
    {"DFMC_Foliage": 4, "DFMC_Wood": 5},
    forecast_date,
    template_path=surface_template,
)
write_grib_dataset(
    dm,
    f"data/DFMC_timemean_{date_string}_FC.grib2",
    {"DFMC_Foliage": 4, "DFMC_Wood": 5},
    forecast_date,
    template_path=surface_template,
)

restart_parameter_ids = {"DFMC_1h": 6, "DFMC_10h": 7, "DFMC_100h": 8, "DFMC_1000h": 9}
restart_filename_prefixes = {"DFMC_1h": "DFMC1", "DFMC_10h": "DFMC10", "DFMC_100h": "DFMC100", "DFMC_1000h": "DFMC1000"}
for z in range(len(time) - 1):
    output_date = forecast_start + datetime.timedelta(days=z + 1)
    output_date_string = output_date.strftime("%Y_%m_%d")
    restart_fields = {
        "DFMC_1h": DFMC1[z + 1, :, :],
        "DFMC_10h": DFMC10[z + 1, :, :],
        "DFMC_100h": DFMC100[z + 1, :, :],
        "DFMC_1000h": DFMC1000[z + 1, :, :],
    }
    for variable_name, values in restart_fields.items():
        restart_ds = xr.Dataset(
            {variable_name: (("time", "lat", "lon"), values[np.newaxis, :, :])},
            coords={"time": [0], "lat": lat, "lon": lon},
        )
        restart_path = f"data/{restart_filename_prefixes[variable_name]}_INI_{output_date_string}.grib2"
        write_grib_dataset(
            restart_ds,
            restart_path,
            {variable_name: restart_parameter_ids[variable_name]},
            output_date.strftime("%Y%m%d"),
            template_path=surface_template,
        )
    print(f"Saved GRIB2 initial conditions for {output_date_string}")

This section will download data from the XDS-Preprod.

The data is large and so may take some time to download.

Load Static Data

We will load some CLIMATE data files

In [ ]:
# Open the Climate files
PO_arr = np.squeeze(read_grib(base_path / "CLIMATE/POP_2020.grib", decode_times=False).values)
RD_arr = np.squeeze(read_grib(base_path / "CLIMATE/road_density_2015_agg_r.grib", decode_times=False).values)

Load Dynamic Data

Here we open the data files we retrieved from the XDS and the CDS as well as the Active Fire Maps

We iterate through each day within the dataset and store the into an output NetCDF file.

In [ ]:
forecast_date = forecast_start_date.strftime("%Y%m%d")
forecast_date_string = forecast_start_date.strftime("%Y_%m_%d")
daily_file = f"data/FORCING_DAILY_{forecast_date}.grib2"
dfmc_grib_file = f"data/DFMC_MAP_{forecast_date_string}_FC.grib2"
lfmc_grib_file = f"data/LFMC_MAP_{forecast_date_string}_FC.grib2"

with ExitStack() as stack:
    t2m = first_variable(stack.enter_context(open_grib(daily_file, grib_filter("t2m"))))
    precipitation = first_variable(stack.enter_context(open_grib(daily_file, grib_filter("tp"))))
    rh = first_variable(stack.enter_context(open_grib(daily_file, grib_filter("rh"))))
    ws = first_variable(stack.enter_context(open_grib(daily_file, grib_filter("ws"))))
    # The fuel moisture maps are on the Open Data grid (longitude -180..180)
    df_foliage = to_training_grid(first_variable(stack.enter_context(open_grib(dfmc_grib_file, grib_filter("DFMC_Foliage")))))
    df_wood = to_training_grid(first_variable(stack.enter_context(open_grib(dfmc_grib_file, grib_filter("DFMC_Wood")))))
    lfmc = to_training_grid(first_variable(stack.enter_context(open_grib(lfmc_grib_file, grib_filter("LFMC")))))

    forecast_count = t2m.sizes[time_dim(t2m)]
    time = t2m.valid_time.values
    lat = t2m.latitude.values
    lon = t2m.longitude.values

    fuel_ds = stack.enter_context(open_grib(base_path / "CLIMATE" / "DM_FUEL.grib2"))
    fuel_time_dim = time_dim(fuel_ds["llfl"])
    fuel_time_count = fuel_ds.sizes[fuel_time_dim]
    fuel_reference_year = pd.Timestamp(fuel_ds["valid_time"].values[0]).year
    all_grids = []

    for i in range(forecast_count):
        forecast_day = forecast_start_date + datetime.timedelta(days=i + 1)
        if forecast_day.month == 2 and forecast_day.day == 29:
            fuel_reference_day = datetime.date(fuel_reference_year, 2, 28)
        else:
            fuel_reference_day = forecast_day.replace(year=fuel_reference_year)
        fuel_index = (fuel_reference_day - datetime.date(fuel_reference_year, 1, 1)).days
        if fuel_index >= fuel_time_count:
            raise ValueError(f"DM_FUEL.grib2 has no daily record for {fuel_reference_day}")
        fuel_fields = tuple(
            np.squeeze(fuel_ds[name].isel({fuel_time_dim: fuel_index}).load().values)
            for name in ("llfl", "lwfl", "dlfl", "dwfl")
        )
        FU_LL, FU_LW, FU_DF, FU_DW = fuel_fields

        # Daily means for forecast day i + 1, fuel moisture at 00 UTC of that day
        T2_ = field_at(t2m, i)
        PR_ = field_at(precipitation, i)
        RH_ = field_at(rh, i)
        WS_ = field_at(ws, i)
        DF_ = field_at(df_foliage, i + 1)
        DW_ = field_at(df_wood, i + 1)
        LF_ = field_at(lfmc, i + 1)

        ft = FU_LL + FU_LW + FU_DF + FU_DW
        mask = ft > 0
        feature_arrays = {
            "PR": PR_[mask],
            "T2": T2_[mask],
            "RH": RH_[mask],
            "WS": WS_[mask],
            "FU_LL": FU_LL[mask],
            "FU_LW": FU_LW[mask],
            "FU_DF": FU_DF[mask],
            "FU_DW": FU_DW[mask],
            "DF": DF_[mask],
            "DW": DW_[mask],
            "LF": LF_[mask],
            "PO": PO_arr[mask],
            "RD": RD_arr[mask],
        }
        X_pred = pd.DataFrame(feature_arrays)
        y_proba = model.predict_proba(X_pred)[:, 1]
        fire_prob_grid = np.full(ft.shape, np.nan, dtype=float)
        fire_prob_grid[mask] = y_proba
        all_grids.append(fire_prob_grid)

fire_prob_array = np.stack(all_grids, axis=0)
ds_out = xr.Dataset(
    {"fire_probability": (("time", "latitude", "longitude"), fire_prob_array)},
    coords={"time": time, "latitude": lat, "longitude": lon},
)
prediction_file = f"./outputs/POF_prediction_{forecast_date}.grib2"
prediction_start = (forecast_start_date + datetime.timedelta(days=1)).strftime("%Y%m%d")
write_grib_dataset(ds_out, prediction_file, PARAM_IDS, prediction_start)
print(f"Prediction saved -> {prediction_file}")